# 40 labels

Imports

In [1]:
# Importing required packages
from __future__ import print_function
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn.utils.spectral_norm as spectral_norm
import torch.backends.cudnn as cudnn
import torch.nn.parallel

import torchvision.utils as vutils
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.transforms import transforms as transform
import matplotlib.pyplot as plt
from PIL import Image
from skimage import io
import numpy as np
import pandas as pd


In [3]:
#setting device,i.e GPU or CPU
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print("Running on GPU ->",device)
else:
    device = torch.device("cpu")
    print("Running on CPU ->",device)

Running on GPU -> cuda:0


#Dataset Class

In [4]:
class CustomDataset(Dataset):
    def __init__(self, pd_df, image_file, transform=None):
        self.labels = pd_df
        self.image_dir = image_file
        self.transform = transform or transforms.Compose([
            transforms.Resize(128),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        labels = (torch.tensor(self.labels.iloc[index].values.reshape(1, -1)) + 1) / 2
        labels = labels.view(-1)

        image_path = os.path.join(self.image_dir, f"{index}.jpg")
        image = Image.open(image_path).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return image, labels


In [ ]:
data = pd.read_csv("/content/CelebAMask-HQ-attribute-anno (2).txt", sep=" ")
labels = data[:5000]
dataset = CustomDataset(labels, "/content/CelebA-HQ-256/")


Additional Layers

In [ ]:
#Additional Layers to use in the Generator and Discriminator

class SelfAttention(nn.Module):
    def __init__(self, in_dim):
        super(SelfAttention, self).__init__()
        self.query_conv = spectral_norm(nn.Conv2d(in_dim, in_dim // 8, kernel_size=1))
        self.key_conv = spectral_norm(nn.Conv2d(in_dim, in_dim // 8, kernel_size=1))
        self.value_conv = spectral_norm(nn.Conv2d(in_dim, in_dim, kernel_size=1))
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.size()
        query = self.query_conv(x).view(B, -1, H * W).permute(0, 2, 1)
        key = self.key_conv(x).view(B, -1, H * W)
        attention = F.softmax(torch.bmm(query, key), dim=-1)
        value = self.value_conv(x).view(B, -1, H * W)
        out = torch.bmm(value, attention.permute(0, 2, 1)).view(B, C, H, W)
        out = self.gamma * out + x
        return out

class UpSamplingBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UpSamplingBlock, self).__init__()
        self.upsample = nn.Upsample(scale_factor=2)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.norm = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.upsample(x)
        x = self.conv(x)
        x = self.norm(x)
        x = self.relu(x)
        return x
class ConvBlock(nn.Module):
    def __init__(self,input_chan, num_kernels, kernel_size=(4, 4), strides=2, activation=True):
        super(ConvBlock, self).__init__()

        self.conv = spectral_norm(nn.Conv2d(input_chan,num_kernels , kernel_size, stride=strides, padding=1, bias=False))
        self.batch_norm = nn.BatchNorm2d(num_kernels)
        self.activation = nn.LeakyReLU(0.2) if activation else nn.Identity()

    def forward(self, x):
        x = self.conv(x)
        x = self.batch_norm(x)
        x = self.activation(x)
        return x

class Label_FC(nn.Module):
    def __init__(self):
        super(Label_FC, self).__init__()

        self.fc = nn.Linear(40, 320)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc(x)
        x = self.relu(x)
        return x

Generator

In [ ]:
#Generator

class Generator(nn.Module):
    def __init__(self, label_size, noise_dim):
        super(Generator, self).__init__()
        self.label_fc = nn.Linear(label_size, 512)
        self.leaky_relu = nn.LeakyReLU(negative_slope=0.2)
        self.fc = nn.Linear(612, 8192, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.reshape = nn.Unflatten(1, (4, 4, 512))

        self.up1 = UpSamplingBlock(512, 256)
        self.SAtt1 = SelfAttention(256)
        self.up2 = UpSamplingBlock(256, 128)
        self.SAtt2 = SelfAttention(128)
        self.up3 = UpSamplingBlock(128, 64)
        self.SAtt3 = SelfAttention(64)
        self.up4 = UpSamplingBlock(64, 32)
        self.SAtt4 = SelfAttention(32)
        self.up5 = UpSamplingBlock(32, 16)

        self.conv_final = spectral_norm(nn.Conv2d(16, 3, kernel_size=3, padding=1, bias=False))
        self.tanh = nn.Tanh()

    def forward(self, noise, text_embedding):
            
        labels = self.leaky_relu(self.label_fc(label_size))
        
        x = torch.cat([labels, noise], dim=1)
        x = self.relu(self.fc(x))
        x = self.reshape(x)
        x = x.view(x.shape[0],x.shape[3],x.shape[2],x.shape[1])
        x = self.up1(x)
        x = self.up2(x)
        x = self.up3(x)
        x = self.SAtt3(x)
        x = self.up4(x)
        x = self.up5(x)

        x = self.tanh(self.conv_final(x))
        return x  

        


Discriminator

In [14]:

#Discriminator

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        self.conv1 = spectral_norm(nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1, bias=False))
        self.leaky_relu = nn.LeakyReLU(0.2)

        self.conv_block1 = ConvBlock(64,128, activation=True)
        self.conv_block2 = ConvBlock(128,256, activation=True)
        self.conv_block3 = ConvBlock(256,512, activation=True)

        self.self_attn = SelfAttention(517)

        self.conv2 = spectral_norm(nn.Conv2d(517, 512, kernel_size=1, stride=1, padding=1, bias=False))
        self.batch_norm2 = nn.BatchNorm2d(512)
        self.leaky_relu2 = nn.LeakyReLU(0.2)

        self.flatten = nn.Flatten()
        self.fc = nn.Linear(51200, 1)
        self.sigmoid = nn.Sigmoid()
        self.Label_FC = Label_FC().to(device)

 
    def forward(self, image, labels):
        x = self.conv1(image)
        x = self.leaky_relu(x)
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        labels = self.Label_FC(labels)
        labels = input_layer2.view(labels.shape[0],5,8,8)
        concat = torch.cat((x, labels), dim=1)
        x1 = self.self_attn(concat)
        x1 = self.conv2(x1)
        x1 = self.batch_norm2(x1)
        x1 = self.leaky_relu2(x1)
        x1 = self.flatten(x1)
        x1 = self.fc(x1)
        x1 = self.sigmoid(x1)
        return x1


torch.Size([1, 1])


In [15]:
#Hyperparameters
latent_dim = 100
conditional_dim = 10
batch_size = 128
num_epochs = 250
lr = 0.0002




In [16]:
trian_data_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)



Plotting the fake images


In [17]:


def plot_fake_images(images, epoch):
    
    if images.is_cuda:
        images = images.cpu()

    # Rescale pixel values to [0, 1]
    images = (images + 1) / 2  

    # Create a grid of images
    grid = vutils.make_grid(images, normalize=False, nrow=8)

    # Plot the grid
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis('off')
    plt.title(f'Fake Images - Epoch {epoch}')
    plt.show()


# Training

In [ ]:

# Loss functions
adversarial_loss = nn.BCELoss()
conditional_loss = nn.CrossEntropyLoss()


# Initialize the generator and discriminator
generator = Generator(label_size=40, noise_dim=latent_dim).to(device)
discriminator = Discriminator().to(device)

# Define the optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

# Start the training loop
for epoch in range(0, num_epochs):
    for i, (real_images, conditions) in enumerate(trian_data_loader):
        batch_size = real_images.size(0)

        # Adversarial ground truths
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # Train the discriminator
        optimizer_D.zero_grad()
        conditions = conditions.to(device)
        real_images = real_images.to(device)
        # Generate fake samples
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator(noise, conditions).detach()

        # Compute discriminator loss on real and fake sampless
        real_outputs = discriminator(real_images, conditions)
        discriminator_real_loss = adversarial_loss(real_outputs, real_labels)
        fake_outputs = discriminator(fake_images, conditions)
        discriminator_fake_loss = adversarial_loss(fake_outputs, fake_labels)

        # Total discriminator loss
        discriminator_loss = discriminator_real_loss + discriminator_fake_loss
        discriminator_loss.backward()
        optimizer_D.step()

        # Train the generator
        optimizer_G.zero_grad()

        # Generate fake samples
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_images = generator(noise, conditions)

        # Compute generator loss
        outputs = discriminator(fake_images, conditions)
        generator_loss = adversarial_loss(outputs, real_labels)
        generator_loss.backward()
        optimizer_G.step()

        # Print training stats
        if i % 300 == 0:
            print(f"[Epoch {epoch}/{num_epochs}] [Batch {i}/{len(trian_data_loader)}] [Discriminator loss: {discriminator_loss.item():.4f}] [Generatorloss: {generatorg_loss.item():.4f}]")
            plot_fake_images(fake_images.detach(), epoch)


In [24]:
#Saving the Model
check_point = {
  'gen_optimizer':optimizer_G.state_dict(),
  'disc_optimizer':optimizer_D.state_dict(),
  'generator':generator.state_dict(),
  'discriminator':discriminator.state_dict()
}
torch.save(check_point, '/content/drive/MyDrive/epoch250_40features.pth')

# Test

In [ ]:
#Converting text into labels
def text_to_list(text):
  text = text.replace('he','male')
  text = text.replace('his','male')
  text = text.replace('him','male')
  text = text.replace('man','male')
  text = text.replace('boy','young male')
  text = text.replace('no hair','bald')
  text = text.replace('good looking', 'attractive')
  text = text.replace('good-looking', 'attractive')
  text = text.replace('good look', 'attractive')
  text = text.replace('eyebrows', 'arched eyebrows')
  text = text.replace('accessory', 'necklace earrings')
  text = text.replace('tired', 'bags under eyes')
  text = text.replace('blonde hair', 'blond hair')
  text = text.replace('brunette hair', 'brown hair')
  text = text.replace('glasses', 'eyeglasses')
  text = text.replace('facial hair', 'goatee mustache sideburns')
  text = text.replace('grey hair', 'gray hair')
  text = text.replace('defined cheekbones', 'high cheekbones')
  text = text.replace('oval-shaped face', 'oval face')
  text = text.replace('sharp nose', 'pointy nose')
  text = text.replace('balding', 'receding hairline')
  text = text.replace('curly hair', 'wavy')
  text = text.replace('headwear', 'hat')
  text = text.replace('makeup', 'lipstick')
  text = text.replace('full lips', 'big lips')
  text = text.replace('prominent nose', 'big nose')
  text = text.replace('dark hair', 'black hair')
  text = text.replace('out of focus', 'blurry')
  text = text.replace('thick eyebrows', 'bushy eyebrows')
  text = text.replace('plump', 'chubby')
  text = text.replace('extra chin', 'double chin')
  text = text.replace('bold makeup', 'heavy makeup')
  text = text.replace('slightly open mouth', 'mouth slightly open')
  text = text.replace('small eyes', 'narrow eyes')
  text = text.replace('clean-shaven', 'no beard')
  text = text.replace('clean shaven', 'no beard')
  text = text.replace('fair skin', 'pale skin')
  text = text.replace('flushed cheeks', 'rosy cheeks')
  text = text.replace('happy', 'smiling')
  text = text.replace('straight hair', 'straight')
  text = text.replace('youthful', 'young')

  labels = [
        '5 o clock shadow', 'arched eyebrows', 'attractive', 'bags under eyes', 'bald',
        'bangs', 'big lips', 'big nose', 'black hair', 'blond hair', 'blurry', 'brown hair',
        'bushy eyebrows', 'chubby', 'double chin', 'eyeglasses', 'goatee', 'gray hair',
        'heavy makeup', 'high cheekbones', 'male', 'mouth slightly open', 'mustache',
        'narrow eyes', 'no beard', 'oval face', 'pale skin', 'pointy nose', 'receding hairline',
        'rosy cheeks', 'sideburns', 'smiling', 'straight', 'wavy', 'earrings',
        'hat', 'lipstick', 'necklace', 'necktie', 'young'
  ]
  output_list = np.zeros(len(labels))
  for i, label in enumerate(labels):
    if label in text:
      output_list[i] = 1
    else:
       output_list[i] = -1

  return output_list

In [27]:
# Generate random noise
noise_dim = 100
noise = torch.randn(1, noise_dim).to(device)
#
Face_discription= input("Please enter a descreition: ")
conditions=text_to_list(Face_discription)

generated_image = generator(noise, conditions)


In [ ]:
#Plotting the generated image
def plot_generated_image(generated_image):
    # Denormalize the generated image
    generated_image_dn = ((generated_image + 1) / 2) * 256

    # Resize the image
    generated_image_dn = F.interpolate(generated_image_dn, size=(250, 250), mode='bicubic', align_corners=False)

    # Convert the tensor to a PIL image
    generated_image_dn = generated_image_dn.squeeze(0).permute(1, 2, 0).byte().cpu().numpy()
    pil_image = Image.fromarray(generated_image_dn)

    # Plot the image
    plt.figure(figsize=(6, 6))
    plt.imshow(pil_image)
    plt.axis('off')
    plt.show()
plot_generated_image(generated_image)
